# 02. Manuscript figures

Every figure reads the processed data (`results/processed`) and the optimization results (`results/optimization`) produced by `scripts/run_pipeline.py`.
Figures are written to `figures/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import pearsonr

from tbopt import config, data, model, optimize, analysis
from tbopt.plotting import (style, panel_label, savefig, COLORS, MARKERS, AGE_COLORS,
                            T_LABELS, C_A0, C_AMIN, C_GRID)
style()
AGES = config.AGE_GROUPS_40
YEARS = config.YEARS

## Fig. 1(a) — fatality rate by age group (2022)

Panel (b), the map of the average age, needs the district boundary files in `data_aux/korea_map/` (not yet included).

In [ ]:
df = data.load_processed(2022)
phi = pd.DataFrame({lab: df[f"phi_{a}"] for a, lab in zip(AGES, config.AGE_LABELS_40)})

fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=phi, width=0.5, showfliers=False, palette=AGE_COLORS, ax=ax)
ax.set_ylim(-0.02, 0.44)
ax.set_xlabel("age group"); ax.set_ylabel(r"$\phi_t$")
panel_label(ax, "a")
savefig(fig, "fig1a_phi_age.pdf")

## Fig. 3 — province totals N, D, H by year

In [ ]:
N = data.load_sido_patients().groupby("sido", sort=False)[list(YEARS)].sum()
D = data.load_sido_fatalities().groupby("sido", sort=False)[list(YEARS)].sum()
H = data.load_sido_hospitals().set_index("sido")[list(YEARS)]
sidos = [s for s in H.index if s != "Sejong"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, tab, lab, letter in zip(axes, [N, D, H], ["$N$", "$D$", "$H$"], "abc"):
    for i, s in enumerate(H.index):
        if s == "Sejong":
            continue
        ax.plot(YEARS, tab.loc[s], marker=MARKERS[i], c=COLORS[i], mec="k", ms=10, label=s)
    ax.set_xlabel("year"); ax.set_ylabel(lab)
    ax.tick_params(axis="x", rotation=60)
    panel_label(ax, letter)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=8, fontsize=12)
fig.tight_layout(rect=[0, 0.05, 1, 1])
savefig(fig, "fig3_hND_sido_year.pdf")

## Fig. 4 — age fractions of patients (n), deaths (d) and population (g), 2022

In [ ]:
n, d = data.sido_age_fractions(2022)
pop = data.load_sido_population_2022().drop(index="Sejong")
g = pop.div(pop.sum(axis=1), axis=0)
H_order = list(data.load_sido_hospitals().sido)
labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "\u226580"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, tab, lab, letter in zip(axes, [n, d, g], ["$n$", "$d$", "$g$"], "abc"):
    for s in tab.index:
        i = H_order.index(s)
        ax.plot(labels, tab.loc[s].values, marker=MARKERS[i], c=COLORS[i], mec="k", ms=10, label=s)
    ax.set_xlabel("age group"); ax.set_ylabel(lab)
    ax.tick_params(axis="x", rotation=60)
    panel_label(ax, letter)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=8, fontsize=12)
fig.tight_layout(rect=[0, 0.05, 1, 1])
savefig(fig, "fig4_RPRNRD.pdf")

## Fig. 5 — age vs. no-age optimum (2022) and minimum fatalities

(a) map: requires boundary files (pending). (b–e) below.

In [ ]:
year = 2022
df = data.load_processed(year)
dy = analysis.hospital_change_age_vs_noage(year)          # (eta_age - eta_x)/eta, a = 0
x_rho = analysis.rho(df)["80_"]
x_ret = analysis.rho_over_eta_tilde(df)["80_"]
E_curve = optimize.E_min_curve(year)
E_t = analysis.E_min_by_age(year)
red = analysis.reduction_summary()
i_min, i_max = analysis.a_bounds(df)
A = config.A_GRID

fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 2, wspace=0.35, hspace=0.45)

ax = fig.add_subplot(gs[0, 0])
ax.scatter(x_rho, dy, ec="k", alpha=0.5, s=150, c="#17DB96")
ax.set_xscale("log"); ax.set_ylim(-0.103, 0.103)
ax.set_xlabel(r"$\rho_{s,t=5}$"); ax.set_ylabel(r"$\frac{\eta^\mathrm{(opt)}_\mathrm{age}-\eta^\mathrm{(opt)}_\times}{\eta}$")
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5, prune="both"))
panel_label(ax, "b", (-0.25, 1.1))

ax = fig.add_subplot(gs[0, 1])
ax.scatter(x_ret, dy, ec="k", alpha=0.5, s=150, c="#EE9EAF")
ax.set_xscale("log"); ax.set_ylim(-0.103, 0.103)
ax.set_xlabel(r"$\rho_{s,t=5}/\tilde\eta_{s,t=5}$"); ax.set_ylabel(r"$\frac{\eta^\mathrm{(opt)}_\mathrm{age}-\eta^\mathrm{(opt)}_\times}{\eta}$")
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5, prune="both"))
panel_label(ax, "c", (-0.25, 1.1))

ax = fig.add_subplot(gs[1, 0])
ax.scatter(A, E_curve.values, c="w", s=120, ec=C_GRID)
ax.scatter(A[config.A_ZERO_INDEX], E_curve.iloc[config.A_ZERO_INDEX], ec="k", s=140, c=C_A0, lw=2)
ax.scatter(A[i_min], E_curve.iloc[i_min], marker="h", ec="k", s=200, c=C_AMIN, lw=2)
ax.set_xlim(-1.45, 0.35); ax.set_ylim(727, 774)
ax.set_xlabel("$a$"); ax.set_ylabel(r"$E^{\mathrm{(min)}}_\mathrm{fatalities}(a)$")
panel_label(ax, "d", (-0.25, 1.1))
ins = inset_axes(ax, width="35%", height="35%", loc="upper center", borderpad=1.2)
for k, a in enumerate(AGES):
    ins.plot(A, E_t[a].values, color=COLORS[k])
ins.set_xlim(A[i_min] - 0.05, A[i_max] + 0.05); ins.set_yscale("log")
ins.set_ylabel(r"$E^\mathrm{(min)}_t$", size=12); ins.tick_params(labelsize=8)

ax = fig.add_subplot(gs[1, 1])
ax.plot(YEARS, red["reduction_a0"], marker="o", mfc=C_A0, mew=1.5, c="k", ls="--", lw=2, ms=12, label="$a=0$")
ax.plot(YEARS, red["reduction_amin"], marker="h", mfc=C_AMIN, mew=1.5, c="k", ls="--", lw=2, ms=12, label=r"$a=a_\mathrm{min}$")
ax.set_xlabel("year"); ax.set_ylabel(r"$\frac{D-E^{\mathrm{(min)}}_\mathrm{fatalities}(a)}{D}$")
ax.legend(loc=1, fontsize="large")
panel_label(ax, "e", (-0.25, 1.1))
savefig(fig, "fig5_bcde.png")

## Fig. S1 — $\eta^{(opt)}_{age}/\eta$ vs $\rho_{s,t}/\tilde\eta_{s,t}$ (a = 0)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, year, letter in zip(axes.flat, YEARS, "abcdefghi"):
    df = data.load_processed(year)
    ratio = optimize.load_result("age", year, config.A_ZERO_INDEX)["h"] / df["h"]
    x = analysis.rho_over_eta_tilde(df)
    for k, a in enumerate(AGES):
        ax.scatter(x[a], ratio, ec="k", c=COLORS[k], alpha=0.3, label=T_LABELS[k])
    ax.set_xscale("log")
    ax.set_xlabel(r"$\rho_{s,t}/\tilde\eta_{s,t}$", size=18); ax.set_ylabel(r"$\eta^\mathrm{(opt)}_\mathrm{age}/\eta$", size=18)
    panel_label(ax, letter, (-0.11, 1.05))
axes.flat[-1].legend()
fig.tight_layout()
savefig(fig, "fig_sm_1.pdf")

## Fig. S2 — age vs. no-age difference vs rescaled density (t = 5 and no-age)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
corr = []
for ax, year, letter in zip(axes.flat, YEARS, "abcdefghi"):
    df = data.load_processed(year)
    y = analysis.hospital_change_age_vs_noage(year)
    x5 = analysis.rho_over_eta_tilde(df)["80_"]
    xt = analysis.rho_over_eta_tilde_total(df)
    ax.scatter(x5, y, ec="k", alpha=0.5, c=COLORS[4], label="$t=5$")
    ax.scatter(xt, y, ec="k", alpha=0.5, c="#BB0D0D", label=r"$\times$ (no-age)")
    ax.set_xscale("log")
    ax.set_xlabel(r"$\rho_{s,t}/\tilde\eta_{s,t}$", size=18)
    ax.set_ylabel(r"$\frac{\eta^\mathrm{(opt)}_\mathrm{age}-\eta^\mathrm{(opt)}_\times}{\eta}$", size=18)
    panel_label(ax, letter, (-0.15, 1.05))
    corr.append((year, pearsonr(x5, y)[0], pearsonr(xt, y)[0]))
axes.flat[6].legend(fontsize="large")
fig.tight_layout()
savefig(fig, "fig_sm_rhotilde_vs_diff.pdf")
pd.DataFrame(corr, columns=["year", "r(t=5)", "r(no-age)"]).set_index("year").round(3)

## Fig. S3 — $E^{(min)}(a)$ for every year

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, year, letter in zip(axes.flat, YEARS, "abcdefghi"):
    df = data.load_processed(year)
    E = optimize.E_min_curve(year)
    i_min, i_max = analysis.a_bounds(df)
    ax.scatter(A, E.values, c="w", s=80, ec=C_GRID)
    ax.scatter(A[i_min], E.iloc[i_min], marker="s", ec="k", s=100, c=COLORS[0], label=r"$a_\mathrm{min}$")
    ax.scatter(A[i_max], E.iloc[i_max], marker="h", ec="k", s=120, c=C_A0, label=r"$a_\mathrm{max}$")
    ax.scatter(A[config.A_ZERO_INDEX], E.iloc[config.A_ZERO_INDEX], marker="o", ec="k", s=100, c=COLORS[1], label="$a=0$")
    ax.set_xlim(A[i_min] - 0.15, A[i_max] + 0.15)
    sel = E.iloc[i_min:i_max + 1]
    pad = 0.1 * (sel.max() - sel.min())
    ax.set_ylim(sel.min() - pad, sel.max() + pad)
    ax.set_title(str(year)); ax.set_xlabel("$a$", size=18); ax.set_ylabel(r"$E^{\mathrm{(min)}}_\mathrm{fatalities}$", size=18)
    panel_label(ax, letter, (-0.15, 1.05))
axes.flat[0].legend()
fig.tight_layout()
savefig(fig, "fig_sm_a_vs_Emin_a.pdf")

## Fig. S4 — change in allocation vs change in fatalities ($a\simeq a_{max}$ vs $a=0$, 2022)

In [ ]:
year = 2022
df = data.load_processed(year)
i_min, i_max = analysis.a_bounds(df)
h0 = optimize.load_result("age", year, config.A_ZERO_INDEX)["h"]
hmax = optimize.load_result("age", year, i_max)["h"]
E0_t = analysis.E_by_age(df, h0, 0.0)
Emax_t = analysis.E_by_age(df, hmax, A[i_max])
dH = (hmax - h0) / df["h"]
dE = (Emax_t.sum(1) - E0_t.sum(1)) / df["D"]
dE_t = (Emax_t - E0_t) / df["D"].to_numpy()[:, None]

fig, (ax, ax1) = plt.subplots(1, 2, figsize=(10, 4))
ax.axhline(0, c="k", lw=0.3); ax.axvline(0, c="k", lw=0.3)
ax.scatter(dH, dE, alpha=0.7, ec="k", s=120, c="#9DBB64")
ax.set_xlabel(r"$\frac{\eta_s^{\rm (opt)}(a\simeq a_{\rm max})}{\eta_s}-\frac{\eta_s^{\rm (opt)}(a=0)}{\eta_s}$", size=18)
ax.set_ylabel(r"$\frac{E_s^{\rm (opt)}(a\simeq a_{\rm max})-E_s^{\rm (opt)}(a=0)}{D_s}$", size=18)
panel_label(ax, "a")

bins = np.linspace(-2.5, 2.0, 80)
centers = 0.5 * (bins[:-1] + bins[1:])
for k, z in enumerate([2, 4, 3, 1, 5]):
    counts, _ = np.histogram(dE_t[:, k], bins=bins)
    ax1.plot(centers, counts, label=T_LABELS[k], lw=2, marker=MARKERS[k], mec="k", ms=10, color=COLORS[k], zorder=z, alpha=0.8)
ax1.set_xlim(-1.6, 1.3)
ax1.set_xlabel(r"$\frac{E_{s,t}^{\rm (opt)}(a\simeq a_{\rm max})-E_{s,t}^{\rm (opt)}(a=0)}{D_s}$", size=18)
ax1.set_ylabel("count", size=18); ax1.legend()
panel_label(ax1, "b")
fig.tight_layout()
savefig(fig, "fig_sm_4.pdf")